# Práctica Final - Versión Mejorada
### Automatización de respuestas a emails de devolución
**Componentes Intergalácticos Industriales S.A.**

Esta versión utiliza:
- `PydanticOutputParser` → reemplaza la función `convertir_a_diccionario()` manual
- `SequentialChain` → reemplaza los `invoke` manuales entre pasos

## 1. Instalación de dependencias

In [1]:
%pip install langchain langchain-classic langchain-groq python-dotenv pydantic -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\macdu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Imports y configuración

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os

load_dotenv()

# Opción A (recomendado): crea un archivo .env en la misma carpeta con este contenido:
# GROQ_API_KEY=sk-...

# Opción B: ponla directamente aquí (no subas esto a GitHub)
# os.environ["GROQ_API_KEY"] = "sk-..."

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

## 3. Parser de salida estructurada
Definimos con Pydantic la estructura que esperamos que devuelva el modelo.
El parser genera automáticamente las instrucciones de formato para el prompt.

In [3]:
class InfoEmail(BaseModel):
    pedido: str = Field(description="Número de pedido")
    remitente: str = Field(description="Nombre del remitente")
    motivo: str = Field(description="Motivo principal de la solicitud")

parser = PydanticOutputParser(pydantic_object=InfoEmail)

## 4. Paso 1 — Extracción de información
Extrae el número de pedido, remitente y motivo del email.

Las instrucciones de formato (`format_instructions`) las genera el parser automáticamente y se inyectan en el prompt.

In [4]:
extract_prompt = PromptTemplate(
    input_variables=["email"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
    template="""Extrae la siguiente información del email:
- Número de pedido
- Nombre del remitente
- Motivo principal de la solicitud

{format_instructions}

Email:
\"\"\"{email}\"\"\"
"""
)

extract_chain = LLMChain(llm=llm, prompt=extract_prompt, output_key="info_json")

C:\Users\macdu\AppData\Local\Temp\ipykernel_6272\1531308009.py:16: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  extract_chain = LLMChain(llm=llm, prompt=extract_prompt, output_key="info_json")


## 5. Paso 2 — Evaluación de la solicitud
Recibe el JSON del paso anterior y decide si se ACEPTA o RECHAZA según la política de devoluciones.

In [5]:
eval_prompt = PromptTemplate(
    input_variables=["info_json"],
    template="""Dado este JSON con información del email: {info_json}

Extrae el motivo y decide si se debe ACEPTAR o RECHAZAR la devolución.

✅ ACEPTAR si:
- Defecto de fabricación
- Error en el suministro
- Producto incompleto desde fábrica

❌ RECHAZAR si:
- Daños durante el transporte (si no es responsabilidad de la empresa)
- Manipulación del cliente
- Solicitud fuera de plazo

Responde ÚNICAMENTE con: "ACEPTAR" o "RECHAZAR"
"""
)

eval_chain = LLMChain(llm=llm, prompt=eval_prompt, output_key="decision")

## 6. Paso 3 — Redacción de la respuesta
Recibe el JSON y la decisión de los pasos anteriores y redacta un email formal y empático.

In [6]:
response_prompt = PromptTemplate(
    input_variables=["info_json", "decision"],
    template="""Dado este JSON con información del email: {info_json}
Y la siguiente decisión: {decision}

Redacta una respuesta formal y empática al cliente.

Si la decisión es ACEPTAR:
- Agradécele por contactar.
- Confirma que el reemplazo será procesado.
- Explica que la empresa cubrirá el fallo según la política.

Si la decisión es RECHAZAR:
- Lamenta la situación.
- Explica que no se puede aceptar la devolución según la política.
- Ofrece ayuda adicional si la necesita.

Finaliza con esta firma:
María Fernández
Responsable de Atención al Cliente
Componentes Intergalácticos Industriales S.A.
contacto@cii.com
"""
)

response_chain = LLMChain(llm=llm, prompt=response_prompt, output_key="respuesta")

## 7. SequentialChain — Unión de los 3 pasos
Declaramos qué variable entra (`email`) y cuáles queremos obtener al final.
LangChain pasa automáticamente `info_json` y `decision` entre pasos.

In [7]:
cadena_completa = SequentialChain(
    chains=[extract_chain, eval_chain, response_chain],
    input_variables=["email"],
    output_variables=["info_json", "decision", "respuesta"],
    verbose=True
)

## 8. Caso 1 — Solicitud RECHAZADA (daños en transporte)

In [8]:
email_rechazar = """Asunto: Solicitud de reemplazo por daños en transporte – Pedido #D347-STELLA

Estimado equipo de Componentes Intergalácticos Industriales S.A.,

Me pongo en contacto con ustedes como cliente reciente para comunicar una incidencia
relacionada con el pedido #D347-STELLA, correspondiente a un lote de condensadores de
fluzo modelo FX-88.

Lamentablemente, al recibir el envío, observamos que varios de los condensadores
presentaban daños visibles. Todo indica que la mercancía sufrió una caída durante
el transporte interestelar.

Solicitamos con urgencia el reemplazo inmediato de las unidades defectuosas.

Atentamente,
Darth Márquez
"""

resultado1 = cadena_completa.invoke({"email": email_rechazar})

print("=" * 60)
print("CASO 1 - RECHAZAR")
print("=" * 60)
print("📋 Info extraída:", resultado1["info_json"])
print("⚖️  Decisión:", resultado1["decision"])
print("📧 Respuesta:\n", resultado1["respuesta"])



> Entering new SequentialChain chain...

> Finished chain.
CASO 1 - RECHAZAR
📋 Info extraída: {
  "pedido": "D347-STELLA",
  "remitente": "Darth Márquez",
  "motivo": "Reemplazo de unidades dañadas durante el transporte"
}
⚖️  Decisión: RECHAZAR
📧 Respuesta:
 Estimado/a Darth Márquez,

Lamentamos mucho la situación que ha experimentado con el pedido **D347‑STELLA** y entendemos lo frustrante que resulta recibir unidades dañadas durante el transporte. Apreciamos que nos haya comunicado el motivo del reemplazo y su intención de gestionar la devolución.

Sin embargo, tras revisar la política de devoluciones de Componentes Intergalácticos Industriales S.A., debemos informarle que, en este caso, no es posible aceptar la devolución del producto. La normativa establece que los reemplazos solo pueden procesarse cuando el daño se haya reportado dentro de los 48 horas posteriores a la entrega y siempre que el embalaje original se mantenga intacto, condiciones que no se cumplen en la presente s

## 9. Caso 2 — Solicitud ACEPTADA (defecto de fabricación)

In [9]:
email_aceptar = """Asunto: Devolución por defecto de fabricación – Pedido #XZ901-LUCA

Estimado equipo de Componentes Intergalácticos Industriales S.A.,

Les escribo para informarles que el pedido #XZ901-LUCA, compuesto por una serie de
microprocesadores CU-92, presenta un defecto de fábrica: varias unidades no responden
a la activación básica ni siquiera tras revisión técnica.

Solicito formalmente la devolución o el reemplazo de las unidades defectuosas.

Gracias por su atención.

Atentamente,
Lucía Robles
"""

resultado2 = cadena_completa.invoke({"email": email_aceptar})

print("=" * 60)
print("CASO 2 - ACEPTAR")
print("=" * 60)
print("📋 Info extraída:", resultado2["info_json"])
print("⚖️  Decisión:", resultado2["decision"])
print("📧 Respuesta:\n", resultado2["respuesta"])



> Entering new SequentialChain chain...

> Finished chain.
CASO 2 - ACEPTAR
📋 Info extraída: {
  "pedido": "XZ901-LUCA",
  "remitente": "Lucía Robles",
  "motivo": "Devolución o reemplazo de unidades defectuosas por defecto de fabricación"
}
⚖️  Decisión: ACEPTAR
📧 Respuesta:
 Estimada Lucía Robles,

Muchas gracias por ponerse en contacto con nosotros y por informarnos sobre la incidencia detectada en su pedido **XZ901‑LUCA**. Lamentamos los inconvenientes que le haya causado este defecto de fabricación.

Nos complace comunicarle que hemos **aceptado su solicitud de devolución y reemplazo**. El proceso de sustitución de las unidades defectuosas será gestionado de inmediato y, conforme a nuestra política de garantía, **la empresa cubrirá íntegramente el fallo** sin coste adicional para usted. En los próximos días recibirá la confirmación del envío del nuevo lote y las instrucciones para la devolución de los productos afectados.

Quedamos a su disposición para cualquier consulta adicio